# Phase 1 Dataset Integrity Audit

This notebook presents the measured integrity audit for `data/raw/PonziDataset_20221114.csv`.
It is intentionally thin: reusable loading and validation logic lives in `src/data/`.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

audit_path = PROJECT_ROOT / "results" / "baseline" / "dataset_audit.json"
if not audit_path.exists():
    from src.data.audit import build_dataset_audit, save_dataset_audit

    save_dataset_audit(build_dataset_audit())

audit = json.loads(audit_path.read_text(encoding="utf-8"))
audit["generated_at_utc"]

## Paper Count Check

In [ ]:
pd.DataFrame(
    [
        {
            "measure": "Total contracts",
            "published": audit["published_reference"]["total_contracts"],
            "measured": audit["row_count"],
            "difference": audit["paper_discrepancies"]["row_count_delta"],
        },
        {
            "measure": "Ponzi contracts",
            "published": audit["published_reference"]["ponzi_contracts"],
            "measured": int(audit["class_counts"].get("1", 0)),
            "difference": audit["paper_discrepancies"]["ponzi_count_delta"],
        },
    ]
)

## Schema, Labels, And Nulls

In [ ]:
pd.DataFrame(
    {
        "column": audit["columns"],
        "dtype": [audit["dtypes"][column] for column in audit["columns"]],
        "null_count": [audit["null_counts"][column] for column in audit["columns"]],
    }
)

In [ ]:
pd.Series(audit["class_counts"], name="count").rename_axis("label").reset_index()

## Integrity Summary

In [ ]:
summary = {
    "duplicate_addresses": audit["duplicate_addresses"],
    "duplicate_transaction_hashes": audit["duplicate_transaction_hashes"],
    "duplicate_block_address_combinations": audit["duplicate_block_address_combinations"],
    "block_number_min": audit["block_number_range"]["min"],
    "block_number_max": audit["block_number_range"]["max"],
    "timestamp_min": audit["timestamp_range"]["min"],
    "timestamp_max": audit["timestamp_range"]["max"],
    "chronological_by_block": audit["chronological_ordering"]["by_createdBlockNumber"],
    "chronological_by_timestamp": audit["chronological_ordering"]["by_timestamp"],
    "unique_creators": audit["unique_creators"],
    "creators_with_multiple_contracts": audit["creators_with_multiple_contracts"],
    "creators_with_multiple_positive_contracts": audit["creators_with_multiple_positive_contracts"],
    "missing_or_empty_opcode_values": audit["opcode"]["missing_or_empty_count"],
    "exact_duplicate_opcode_rows": audit["opcode"]["exact_duplicate_rows"],
    "exact_duplicate_opcode_groups": audit["opcode"]["exact_duplicate_groups"],
}
pd.Series(summary, name="value").rename_axis("check").reset_index()

## Reproducibility

In [ ]:
pd.Series(
    {
        "sha256": audit["sha256"],
        "file_size_bytes": audit["file_size_bytes"],
        "metadata_only_deep_bytes": audit["memory_usage_bytes"]["metadata_only_deep"],
        "opcode_deep_bytes": audit["memory_usage_bytes"]["opcode_deep"],
        "metadata_plus_opcode_deep_bytes": audit["memory_usage_bytes"]["metadata_plus_opcode_deep"],
        "raw_csv_file_size_bytes": audit["memory_usage_bytes"]["raw_csv_file_size"],
    },
    name="value",
).rename_axis("measure").reset_index()